# 01 — Environment and Compute

## Objective
Inspect the environment, calculate memory costs, and reproduce randomness.

## Background
CUDA (Compute Unified Device Architecture) executes supported tensor operations on NVIDIA GPUs (Graphics Processing Units). VRAM (Video Random Access Memory) is limited on consumer cards. Low-bit weight storage reduces model-weight memory but not all optimizer and activation costs.

### Locate output storage

**Procedure.** Find the research directory and create the local run folder.

**Reading the result.** The path printed here determines where later checkpoints and predictions will be saved.

In [1]:
# Locate the research workspace from the repository root or the notebooks folder.
from pathlib import Path
cwd = Path.cwd().resolve()
if (cwd / "exploration" / "notebooks").is_dir():
    RESEARCH = cwd / "exploration"
elif cwd.name == "notebooks" and cwd.parent.name == "exploration":
    RESEARCH = cwd.parent
elif cwd.name == "exploration" and (cwd / "notebooks").is_dir():
    RESEARCH = cwd
else:
    raise FileNotFoundError("Open Jupyter from the repository root, exploration/, or exploration/notebooks/.")

RUNS = RESEARCH / "runs"
RUNS.mkdir(parents=True, exist_ok=True)
print("Research workspace:", RESEARCH)
print("Experiment outputs:", RUNS)

Research workspace: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration
Experiment outputs: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration\runs


### Inspect software and accelerator availability

**Procedure.** Print the Python version, installed package versions, and available Graphics Processing Unit (GPU) information.

**Reading the result.** Missing packages are reported explicitly. GPU and memory values describe this computer, not model requirements.

In [2]:
# Inspect reproducibility-relevant versions before allocating GPU memory.
import sys
import platform
from importlib import metadata

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
for name in ("torch", "transformers", "datasets", "trl", "peft", "bitsandbytes", "accelerate", "scikit-learn"):
    try:
        version = metadata.version(name)
    except metadata.PackageNotFoundError:
        version = "not installed"
    print(f"{name:16} {version}")

try:
    import torch
    available = torch.cuda.is_available()
    print("CUDA available:", available)
    if available:
        print("GPU:", torch.cuda.get_device_name(0))
        print("CUDA runtime:", torch.version.cuda)
        free_bytes, total_bytes = torch.cuda.mem_get_info(0)
        print("Total VRAM (GiB):", round(total_bytes / 2**30, 2))
        print("Free VRAM now (GiB):", round(free_bytes / 2**30, 2))
        # Verify that the detected accelerator executes a small tensor operation.
        probe = torch.tensor([1.0, 2.0], device="cuda:0")
        print("CUDA computation check:", (probe * 2).tolist())
except ImportError:
    print("PyTorch is not installed; GPU information is unavailable.")

Python: 3.10.15
Platform: Windows-10-10.0.26300-SP0
torch            2.4.1
transformers     5.0.0
datasets         5.0.1
trl              1.14.0
peft             0.20.0
bitsandbytes     0.50.2
accelerate       1.12.0
scikit-learn     1.5.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4070
CUDA runtime: 12.4
Total VRAM (GiB): 11.99
Free VRAM now (GiB): 10.85
CUDA computation check: [2.0, 4.0]


### Parameter memory estimate
A rough estimate for unquantized weights is

$$M_{\mathrm{weights}} \approx N_{\mathrm{parameters}} \times b/8$$

where $b$ is bits per weight. This ignores activation, optimizer, gradient, quantization-metadata, and temporary memory.

### Estimate parameter-weight memory

**Procedure.** Calculate the ideal weight storage for a 1.5-billion-parameter model at several precisions and seed pseudorandom generators.

**Reading the result.** Values in gibibytes (GiB) exclude gradients, optimizer states, activations, quantization overhead, and caches.

In [3]:
# A theoretical parameter-size estimate excludes activations, optimizer states,
# temporary tensors, optimizer-specific metadata, and inference KV cache.
import random
import numpy as np

parameter_count = 1_500_000_000
for precision_bits in (32, 16, 8, 4):
    weight_gib = parameter_count * precision_bits / 8 / 2**30
    print(f"{precision_bits:2d}-bit parameters: {weight_gib:.2f} GiB (weights only)")

random.seed(42)
np.random.seed(42)
try:
    torch.manual_seed(42)
except NameError:
    pass
print("Pseudorandom generators seeded at 42; deterministic GPU kernels may require additional setup.")

32-bit parameters: 5.59 GiB (weights only)
16-bit parameters: 2.79 GiB (weights only)
 8-bit parameters: 1.40 GiB (weights only)
 4-bit parameters: 0.70 GiB (weights only)
Pseudorandom generators seeded at 42; deterministic GPU kernels may require additional setup.


## Key observations and technical discussion
A model that fits for inference may not fit for training. Record the actual GPU, precision, maximum sequence length, batch size, peak memory, and runtime for every experiment.

## Q&A
**Question:** Why is a 4-bit parameter estimate not sufficient to predict training memory?

**Answer:** Gradients, optimizer state, activations, attention buffers, and quantization metadata require additional memory.

### Reproducibility note
Each printed example belongs to one of three categories: an illustrative unit check, a source-backed inspection, or an opt-in model experiment. Only real executed experiments with source/model identifiers and saved outputs can support research findings.